In [2]:
import duckdb
from pathlib import Path

DB_PATH = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb")

con = duckdb.connect(str(DB_PATH))

con.execute("""
CREATE OR REPLACE TABLE features_tax_by_acct AS
SELECT
    acct,
    COUNT(DISTINCT tax_year) AS delinquent_years,
    MIN(tax_year) AS first_delinquent_year,
    MAX(tax_year) AS most_recent_delinquent_year,
    COUNT(*) AS delinquent_jurisdiction_records
FROM tax_delinquency_events
GROUP BY acct
""")

print(con.sql("SELECT * FROM features_tax_by_acct LIMIT 10"))

con.close()

┌───────────────┬──────────────────┬───────────────────────┬─────────────────────────────┬─────────────────────────────────┐
│     acct      │ delinquent_years │ first_delinquent_year │ most_recent_delinquent_year │ delinquent_jurisdiction_records │
│    varchar    │      int64       │         int64         │            int64            │              int64              │
├───────────────┼──────────────────┼───────────────────────┼─────────────────────────────┼─────────────────────────────────┤
│ 6000000962172 │               10 │                  2015 │                        2024 │                              10 │
│ 6000000962224 │                6 │                  2019 │                        2024 │                               6 │
│ 6000000962277 │                2 │                  2023 │                        2024 │                               2 │
│ 6000000962545 │                5 │                  2020 │                        2024 │                               5 │


In [5]:
import duckdb
from pathlib import Path

DB_PATH = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb")

con = duckdb.connect(str(DB_PATH))
con.execute("""
CREATE OR REPLACE TABLE features_tax_by_acct AS
SELECT
    acct,
    COUNT(DISTINCT tax_year) AS delinquent_years,
    MIN(tax_year) AS first_delinquent_year,
    MAX(tax_year) AS most_recent_delinquent_year,
    2024 - MAX(tax_year) AS years_since_last_delinquency,
    COUNT(*) AS delinquent_jurisdiction_records
FROM tax_delinquency_events
GROUP BY acct
""")
con.close()

In [6]:
import duckdb
from pathlib import Path

DB_PATH = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb")

con = duckdb.connect(str(DB_PATH))
con.execute("""
ALTER TABLE features_tax_by_acct
ADD COLUMN delinquency_span_years INT
""")

con.execute("""
UPDATE features_tax_by_acct
SET delinquency_span_years =
    most_recent_delinquent_year - first_delinquent_year
""")
con.close()

In [10]:
import duckdb
from pathlib import Path

DB_PATH = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb")

con = duckdb.connect(str(DB_PATH))

print(con.sql("""
SELECT
    COUNT(*) AS delinquent_properties
FROM features_tax_by_acct
"""))

con.close()

┌───────────────────────┐
│ delinquent_properties │
│         int64         │
├───────────────────────┤
│                 67307 │
└───────────────────────┘



In [11]:
import duckdb
from pathlib import Path

DB_PATH = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb")

con = duckdb.connect(str(DB_PATH))

con.execute("""
ALTER TABLE features_tax_by_acct
ADD COLUMN currently_delinquent BOOLEAN
""")

con.execute("""
UPDATE features_tax_by_acct
SET currently_delinquent =
    most_recent_delinquent_year = 2024
""")

con.close()

In [12]:
import duckdb
from pathlib import Path

DB_PATH = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb")

con = duckdb.connect(str(DB_PATH))

con.execute("""
ALTER TABLE features_tax_by_acct
ADD COLUMN newly_delinquent BOOLEAN
""")

con.execute("""
UPDATE features_tax_by_acct
SET newly_delinquent =
    first_delinquent_year >= 2023
""")

con.close()

In [13]:
import duckdb
from pathlib import Path

DB_PATH = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb")

con = duckdb.connect(str(DB_PATH))
con.execute("""
ALTER TABLE features_tax_by_acct
ADD COLUMN chronic_delinquency BOOLEAN
""")

con.execute("""
UPDATE features_tax_by_acct
SET chronic_delinquency =
    delinquent_years >= 5
""")

con.close()

In [14]:
import duckdb
from pathlib import Path

DB_PATH = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb")

con = duckdb.connect(str(DB_PATH))
print(con.sql("""
SELECT
    currently_delinquent,
    COUNT(*) AS properties
FROM features_tax_by_acct
GROUP BY currently_delinquent
"""))

con.close()

┌──────────────────────┬────────────┐
│ currently_delinquent │ properties │
│       boolean        │   int64    │
├──────────────────────┼────────────┤
│ false                │       9588 │
│ true                 │      57719 │
└──────────────────────┴────────────┘



In [15]:
import duckdb
from pathlib import Path

DB_PATH = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb")

con = duckdb.connect(str(DB_PATH))
print(con.sql("""
SELECT
    first_delinquent_year,
    COUNT(*) AS properties
FROM features_tax_by_acct
GROUP BY first_delinquent_year
ORDER BY first_delinquent_year DESC
LIMIT 10
"""))

con.close()

┌───────────────────────┬────────────┐
│ first_delinquent_year │ properties │
│         int64         │   int64    │
├───────────────────────┼────────────┤
│                  2024 │      25637 │
│                  2023 │      10886 │
│                  2022 │       6231 │
│                  2021 │       3474 │
│                  2020 │       3636 │
│                  2019 │       1901 │
│                  2018 │       1318 │
│                  2017 │       1061 │
│                  2016 │        863 │
│                  2015 │        803 │
├───────────────────────┴────────────┤
│ 10 rows                  2 columns │
└────────────────────────────────────┘



In [16]:
import duckdb
from pathlib import Path

DB_PATH = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb")

con = duckdb.connect(str(DB_PATH))
print(con.sql("""
SELECT
    delinquent_years,
    COUNT(*) AS properties
FROM features_tax_by_acct
GROUP BY delinquent_years
ORDER BY delinquent_years DESC
LIMIT 10
"""))

con.close()

┌──────────────────┬────────────┐
│ delinquent_years │ properties │
│      int64       │   int64    │
├──────────────────┼────────────┤
│               45 │          2 │
│               41 │          2 │
│               39 │          4 │
│               37 │          2 │
│               36 │          2 │
│               35 │          3 │
│               33 │          4 │
│               32 │          8 │
│               31 │          3 │
│               30 │          3 │
├──────────────────┴────────────┤
│ 10 rows             2 columns │
└───────────────────────────────┘



In [17]:
import duckdb
from pathlib import Path

DB_PATH = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\real_estate.duckdb")

con = duckdb.connect(str(DB_PATH))
print(con.sql("""
SELECT
    COUNT(*) AS properties_3yr_delinquent
FROM features_tax_by_acct
WHERE delinquent_years >= 3
"""))

con.close()

┌───────────────────────────┐
│ properties_3yr_delinquent │
│           int64           │
├───────────────────────────┤
│                     24465 │
└───────────────────────────┘

